# Telecom Customer Churn Prediction & Retention Intelligence System
**Complete Google Colab project | Python · Pandas · scikit-learn · SHAP · Streamlit**

**Goal:** Predict customer churn, explain key drivers, identify at-risk customers, and generate practical retention suggestions. Work through the cells **in order**. All modeling uses a held-out test set and training-only preprocessing to reduce leakage.

**Important:** This is a learning prototype, not a production-ready retention decision system. Retention actions are illustrative business rules, not proven causal interventions.

## Phase 1 — Environment and reproducibility
Colab's standard CPU runtime is sufficient. The first cell installs SHAP if needed; the rest uses common Colab libraries.

In [ ]:
%pip -q install shap joblib

In [ ]:
import sys, warnings, json, os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
                             PrecisionRecallDisplay)
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
print('Python:', sys.version.split()[0], '| scikit-learn:', sklearn.__version__)

## Phase 2 — Load and understand the dataset
We use the public **IBM Telco Customer Churn** sample dataset. The URL below points to a public GitHub-hosted copy. If the URL fails, upload a CSV named `WA_Fn-UseC_-Telco-Customer-Churn.csv` using the Colab file picker. **Do not use real customer personal data** without authorization.

Dataset contains a `Churn` target (`Yes`/`No`). `customerID` is an identifier, not a predictive feature.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
try:
    df = pd.read_csv(DATA_URL)
    print('Loaded sample dataset from public URL')
except Exception as exc:
    print('Download failed:', exc)
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Please upload the IBM Telco churn CSV to continue.')
    df = pd.read_csv(next(iter(uploaded)))
print('Shape:', df.shape)
display(df.head())

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData types:')
display(df.dtypes.to_frame('dtype'))
print('\nMissing values before conversion:')
display(df.isna().sum().sort_values(ascending=False).head(15))
print('\nTarget distribution:')
display(df['Churn'].value_counts(dropna=False).to_frame('customers'))

## Phase 3 — Cleaning and exploratory data analysis
`TotalCharges` is sometimes stored as text with blanks. We convert invalid entries to missing values; the model pipeline will impute them using **training data only**.

In [ ]:
df = df.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
assert set(df['Churn'].dropna().unique()) <= {'Yes', 'No'}, 'Unexpected target labels'
assert df['customerID'].is_unique, 'Duplicate customer IDs found; investigate before modeling'
print('Missing TotalCharges:', df['TotalCharges'].isna().sum())
print('Duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T.head(25))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.countplot(data=df, x='Churn', ax=axes[0]); axes[0].set_title('Churn distribution')
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1]); axes[1].set_title('Monthly charges by churn')
sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[2]); axes[2].set_title('Tenure by churn')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col in zip(axes, ['Contract', 'InternetService', 'PaymentMethod']):
    rates = df.groupby(col)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, title=f'Observed churn rate: {col}')
    ax.set_ylabel('Churn rate'); ax.set_ylim(0, 1); ax.tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()
print('These are associations, not evidence that a feature causes churn.')

## Phase 4 — Train/test split and preprocessing
**Why split first?** We reserve 20% for a final, untouched evaluation. Imputation, scaling, and encoding are learned only inside the training pipeline. The target is converted to 1 = churn and 0 = stay.

In [ ]:
X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn'].map({'No': 0, 'Yes': 1}).astype(int)
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, df['customerID'], test_size=0.20, stratify=y, random_state=RANDOM_STATE)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Numeric:', num_cols)
print('Categorical:', cat_cols)

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_cols)
], remainder='drop')
print('Preprocessing pipeline ready.')

## Phase 5 — Feature engineering (optional, leakage-safe)
We retain the original features for a clean baseline. We can later test domain features such as number of subscribed services, but should add them **inside a transformer pipeline** and compare using cross-validation. The current notebook prioritizes reproducibility over unnecessary feature complexity.

## Phase 6 — Compare multiple ML algorithms
We compare Logistic Regression and Random Forest with stratified 5-fold cross-validation **on training data only**. ROC-AUC measures ranking ability; average precision is useful when the positive class is less common.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=250, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring={'roc_auc':'roc_auc', 'average_precision':'average_precision', 'f1':'f1'},
                            n_jobs=1)
    rows.append({'model': name, **{k.replace('test_', ''): np.mean(v) for k,v in scores.items() if k.startswith('test_')}})
comparison = pd.DataFrame(rows).sort_values('roc_auc', ascending=False)
display(comparison.round(4))

## Phase 7 — Tune the selected model and evaluate
Hyperparameter search happens **only on the training set**. The test set is inspected once afterward. The 0.50 decision threshold is a baseline; business deployment should validate a threshold using costs and validation data, not tune it on the test set.

In [ ]:
winner = comparison.iloc[0]['model']
base = models[winner]
param_dist = ({'model__C': np.logspace(-2, 2, 12)} if winner == 'Logistic Regression' else
              {'model__max_depth': [4, 6, 8, 12, None], 'model__min_samples_leaf': [2, 3, 5, 8], 'model__max_features': ['sqrt', 0.8]})
search = RandomizedSearchCV(
    Pipeline([('preprocess', preprocessor), ('model', base)]),
    param_distributions=param_dist, n_iter=8, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=1, refit=True)
search.fit(X_train, y_train)
final_model = search.best_estimator_
print('Selected:', winner)
print('Best CV ROC-AUC:', round(search.best_score_, 4))
print('Best params:', search.best_params_)

In [ ]:
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)
metrics = {
    'accuracy': accuracy_score(y_test, test_pred),
    'precision': precision_score(y_test, test_pred, zero_division=0),
    'recall': recall_score(y_test, test_pred, zero_division=0),
    'f1': f1_score(y_test, test_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, test_proba),
    'average_precision': average_precision_score(y_test, test_proba)
}
display(pd.Series(metrics, name='held_out_test_score').round(4).to_frame())
print(classification_report(y_test, test_pred, target_names=['Stay', 'Churn'], zero_division=0))
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, display_labels=['Stay', 'Churn'], ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, test_proba, ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=axes[2])
plt.tight_layout(); plt.show()

## Phase 8 — Explainability with SHAP
SHAP estimates how encoded features contribute to a prediction for this **model**. It does not establish causal relationships. To keep Colab responsive, we explain a small sample of transformed test records using a model-specific explainer. Some SHAP versions have different output shapes; the helper below handles common binary-classification variants.

In [ ]:
import shap
encoded_train = final_model.named_steps['preprocess'].transform(X_train)
encoded_test = final_model.named_steps['preprocess'].transform(X_test)
feature_names = final_model.named_steps['preprocess'].get_feature_names_out()
background = shap.sample(encoded_train, min(100, len(encoded_train)), random_state=RANDOM_STATE)
explain_rows = encoded_test[:min(60, len(encoded_test))]
model_estimator = final_model.named_steps['model']
try:
    if isinstance(model_estimator, LogisticRegression):
        explainer = shap.LinearExplainer(model_estimator, background)
    else:
        explainer = shap.TreeExplainer(model_estimator)
    shap_values = explainer(explain_rows)
    values = np.asarray(shap_values.values)
    if values.ndim == 3:  # binary classifier may return one array per class
        values = values[:, :, 1]
    if values.shape != explain_rows.shape:
        raise ValueError(f'Unexpected SHAP shape {values.shape}')
    importance = pd.Series(np.abs(values).mean(axis=0), index=feature_names).sort_values(ascending=False).head(15)
    importance.sort_values().plot(kind='barh', figsize=(9, 6), title='Top features by mean |SHAP value|')
    plt.xlabel('Mean absolute SHAP contribution'); plt.tight_layout(); plt.show()
    display(importance.to_frame('mean_absolute_shap'))
except Exception as exc:
    print('SHAP visualization unavailable in this runtime:', repr(exc))
    print('The trained model and predictions remain usable. Consider updating SHAP or restarting the runtime.')

## Phase 9 — Customer risk segmentation and retention recommendations
We classify churn probabilities into **illustrative** low/medium/high segments. The thresholds (0.30 and 0.60) are examples, not validated operational cutoffs. Recommendations use transparent rules based on customer attributes; they are **not individualized causal treatment estimates**.

In [ ]:
def risk_band(prob):
    return 'High' if prob >= 0.60 else ('Medium' if prob >= 0.30 else 'Low')

def recommend_action(row):
    actions = []
    if row.get('Contract') == 'Month-to-month':
        actions.append('Offer an optional longer-term plan with transparent pricing')
    if row.get('TechSupport') == 'No':
        actions.append('Offer a technical-support consultation')
    if pd.notna(row.get('MonthlyCharges')) and row['MonthlyCharges'] >= 80:
        actions.append('Review plan fit and lower-cost alternatives')
    if row.get('tenure', 999) <= 12:
        actions.append('Provide proactive onboarding and service check-in')
    return '; '.join(actions) if actions else 'Request feedback and monitor service satisfaction'

results = X_test.copy()
results.insert(0, 'customerID', ids_test.to_numpy())
results['actual_churn'] = y_test.to_numpy()
results['churn_probability'] = test_proba
results['predicted_churn'] = test_pred
results['risk_segment'] = results['churn_probability'].map(risk_band)
results['retention_suggestion'] = results.apply(recommend_action, axis=1)
display(results[['customerID', 'churn_probability', 'risk_segment', 'retention_suggestion']].sort_values('churn_probability', ascending=False).head(15))
display(results['risk_segment'].value_counts().to_frame('customers'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=results, x='risk_segment', order=['Low', 'Medium', 'High'], ax=axes[0])
axes[0].set_title('Risk segments (held-out customers)')
sns.histplot(data=results, x='churn_probability', bins=20, ax=axes[1])
axes[1].set_title('Predicted churn probability distribution')
plt.tight_layout(); plt.show()

## Phase 10 — Export the model and results
We save the complete preprocessing + prediction pipeline as `joblib`, plus the scored customer table and a small JSON report. **Security:** Only load model files you created or trust; joblib/pickle files can execute code when loaded. Do not publish real customer-level data.

In [ ]:
out = pathlib.Path('telecom_churn_outputs'); out.mkdir(exist_ok=True)
joblib.dump(final_model, out / 'churn_pipeline.joblib')
results.to_csv(out / 'customer_risk_predictions.csv', index=False)
with open(out / 'model_report.json', 'w') as f:
    json.dump({'selected_model': winner, 'best_params': search.best_params_, 'test_metrics': metrics,
               'feature_columns': X_train.columns.tolist(), 'risk_thresholds': {'medium': 0.30, 'high': 0.60}},
              f, indent=2, default=str)
print('Saved:', [p.name for p in out.iterdir()])

## Phase 11 — Starter Streamlit dashboard
The next cell writes a separate `app.py` file and `requirements.txt`. Download the ZIP from Colab, extract it locally, copy `churn_pipeline.joblib` into the same folder as `app.py`, and run `streamlit run app.py` in an environment with the dependencies installed. **Streamlit is not automatically hosted by Colab.** The dashboard supports uploading a CSV with the same input columns as training data.

In [ ]:
from pathlib import Path
app_source = 'import json\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\nst.set_page_config(page_title="Telecom Churn Intelligence", layout="wide")\nst.title("📡 Telecom Churn & Retention Intelligence")\nst.caption("Educational prototype — predictions and suggestions require business validation")\n\n@st.cache_resource\ndef load_model():\n    return joblib.load("churn_pipeline.joblib")\n\nmodel = load_model()\nwith open("model_report.json", encoding="utf-8") as f:\n    report = json.load(f)\nrequired = report["feature_columns"]\nuploaded = st.file_uploader("Upload customer CSV (same feature columns as training)", type="csv")\nif uploaded is not None:\n    df = pd.read_csv(uploaded)\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        st.error("Missing columns: " + ", ".join(missing))\n    else:\n        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")\n        probs = model.predict_proba(df[required])[:, 1]\n        scored = df.copy()\n        scored["churn_probability"] = probs\n        scored["risk_segment"] = pd.cut(probs, bins=[-0.01, .30, .60, 1.01], labels=["Low", "Medium", "High"], right=False)\n        st.metric("Customers", len(scored))\n        st.metric("High risk", int((probs >= .60).sum()))\n        st.metric("Average predicted churn", f"{probs.mean():.1%}")\n        st.bar_chart(scored["risk_segment"].value_counts())\n        st.dataframe(scored.sort_values("churn_probability", ascending=False), use_container_width=True)\n        st.download_button("Download scored CSV", scored.to_csv(index=False), "scored_customers.csv", "text/csv")\nelse:\n    st.info("Upload a customer CSV to begin. Never upload private customer data without permission.")\n'
(out / 'app.py').write_text(app_source, encoding='utf-8')
(out / 'requirements.txt').write_text('streamlit>=1.40,<2\npandas>=2.2,<3\nnumpy>=1.26,<3\nscikit-learn>=1.5,<2\njoblib>=1.4,<2\n', encoding='utf-8')
import shutil
archive = shutil.make_archive('telecom_churn_complete', 'zip', root_dir=out)
print('Created:', archive)
from google.colab import files
files.download(archive)

## Phase 12 — Reflection and next experiments
1. Compare recall, precision, ROC-AUC and average precision; explain why accuracy alone can mislead.
2. Compare model results across different customer groups, where appropriate and permitted, before any operational use.
3. Tune decision thresholds **using cross-validated or separate validation predictions**, not the held-out test set.
4. Try engineered service-count features inside the pipeline and evaluate whether they improve cross-validation scores.
5. Calibrate probabilities and check calibration curves before interpreting probabilities as business risk.
6. Evaluate retention suggestions in a controlled experiment before claiming they reduce churn.

**Deliverables:** trained model, prediction CSV, evaluation report, Streamlit starter app, and this notebook. Publish only synthetic or authorized data.